# Notebook 2: Experiment 2 — Cross-Stock Prediction (80/20)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on one stock's daily data, predict on another stock's daily test data.  
**Train/Test Split:** 80/20 (chronological)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Scaler:** ProportionScaler (÷ 10,501 BBCA ATH)  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  


In [4]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

set_seed()
set_ieee_style()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.8
RATIO_LABEL = '80_20'
EXP_LABEL = f'Exp2_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 2 - Cross-Stock Prediction (80/20)")


Experiment 2 - Cross-Stock Prediction (80/20)


In [5]:
# Load all daily data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\nAll daily data loaded!")


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

All daily data loaded!


## Run All Cross-Stock Experiments

### Methodology: Zero-Shot Transfer Learning
Instead of retraining from scratch, we load pre-trained models from **Experiment 1** (with same-stock data) and directly apply them to predict different stocks' test data. This tests how well models generalize across stocks without any domain adaptation.

**Expected result:** Performance will likely be worse than same-stock predictions due to different price ranges, volatility, and patterns across stocks. However, this shows raw transfer capability.

In [7]:
# ============================================================
# EXPERIMENT 2: Cross-stock prediction (using Exp1 pre-trained models)
# Load models trained on Stock A, test on Stock B (zero-shot transfer)
# ============================================================
from tensorflow.keras.models import load_model

all_results = []
all_predictions = {}  # {(train_stock, test_stock): {model_type: (y_true, y_pred, dates)}}

for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue  # Skip same-stock (covered in Exp 1)
        
        pair_key = (train_stock, test_stock)
        print(f"\n{'#'*60}")
        print(f"# TRAIN: {train_stock} -> TEST: {test_stock}")
        print(f"# Using pre-trained Exp1 models (zero-shot transfer)")
        print(f"{'#'*60}")
        
        # Prepare cross-stock data
        X_train, y_train, X_test, y_test, test_dates = prepare_cross_stock_data(
            daily_data[train_stock], daily_data[test_stock],
            train_ratio=TRAIN_RATIO, lookback=LOOKBACK
        )
        print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
        
        all_predictions[pair_key] = {}
        
        for model_type in MODEL_TYPES:
            # Build Exp1 model filename
            exp1_model_path = f'models/Exp1_80_20/Exp1_80_20_{train_stock}_{model_type}_best.keras'
            
            if not os.path.exists(exp1_model_path):
                print(f"  ⚠️  Model not found: {exp1_model_path}")
                continue
            
            print(f"\n  Loading {model_type} from: {exp1_model_path}")
            
            # Load pre-trained model
            model = load_model(exp1_model_path)
            
            # Make predictions on test data (NO training)
            y_pred_scaled = model.predict(X_test, verbose=0).flatten()
            
            # Inverse scale to original values
            y_true_inv = proportion_inverse_scale(y_test)
            y_pred_inv = proportion_inverse_scale(y_pred_scaled)
            
            # Evaluate
            metrics = evaluate_predictions(y_true_inv, y_pred_inv)
            
            result = {
                'Train_Stock': train_stock,
                'Test_Stock': test_stock,
                'Model': model_type,
                **metrics
            }
            all_results.append(result)
            all_predictions[pair_key][model_type] = (y_true_inv, y_pred_inv, test_dates)
            
            print(f"    RMSE: {metrics['RMSE']:.4f}, MAE: {metrics['MAE']:.4f}, R²: {metrics['R2']:.6f}")
            
            # Plot prediction
            plot_actual_vs_predicted(
                test_dates, y_true_inv, y_pred_inv,
                model_type, f'Train_{train_stock}_Test_{test_stock}',
                EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
            )

print("\n\nAll Experiment 2 (80/20) cross-stock prediction complete!")



############################################################
# TRAIN: TLKM -> TEST: BBCA
# Using pre-trained Exp1 models (zero-shot transfer)
############################################################
  X_train: (4193, 1, 1), X_test: (1049, 1, 1)

  Loading BiLSTM from: models/Exp1_80_20/Exp1_80_20_TLKM_BiLSTM_best.keras
    RMSE: 279.8613, MAE: 245.6907, R²: 0.927536


MemoryError: bad allocation

## Results Summary

In [ ]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 2 - Cross-Stock Prediction (80/20)")

results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")



  Experiment 2 - Cross-Stock Prediction (80/20)
Train_Stock Test_Stock  Model         MSE     RMSE      MAE  MAPE (%)       R2
       TLKM       BBCA BiLSTM  19793.8181 140.6905 106.9362    1.3046 0.981687
       TLKM       BBCA  BiGRU  23737.7499 154.0706 122.6106    1.4822 0.978038
       TLKM       BBCA   LSTM 100021.7674 316.2622 273.1215    3.2265 0.907459
       TLKM       BBCA    GRU  19989.5070 141.3843 107.0527    1.2926 0.981506
       TLKM       ASII BiLSTM  11630.2970 107.8439  81.3928    1.7041 0.968521
       TLKM       ASII  BiGRU  12470.1760 111.6699  85.7536    1.7888 0.966247
       TLKM       ASII   LSTM  14053.0544 118.5456  88.4360    1.8438 0.961963
       TLKM       ASII    GRU   8559.4745  92.5174  69.2994    1.4749 0.976832
       TLKM       UNVR BiLSTM  10020.8673 100.1043  71.5909    2.6640 0.988480
       TLKM       UNVR  BiGRU   9380.1788  96.8513  71.4492    2.6757 0.989217
       TLKM       UNVR   LSTM  11451.5093 107.0117  71.2939    2.6110 0.986836
   

## Visualizations

In [ ]:
# ============================================================
# HEATMAPS PER MODEL
# ============================================================
for metric in ['RMSE', 'MAE', 'MAPE (%)', 'R2']:
    plot_metrics_heatmap(
        results_df, metric, EXP_LABEL,
        row_col='Train_Stock', col_col='Test_Stock',
        save_dir=f'figures/{EXP_LABEL}'
    )

print("All heatmaps saved!")


  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiLSTM_RMSE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiGRU_RMSE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_LSTM_RMSE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_GRU_RMSE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiLSTM_MAE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiGRU_MAE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_LSTM_MAE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_GRU_MAE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiLSTM_MAPE_pct_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiGRU_MAPE_pct_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_LSTM_MAPE_pct_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_GRU_MAPE_pct_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiLSTM_R2_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiGRU_R2_heatmap.png
  Figure saved: figures/Exp2_80

In [ ]:
# ============================================================
# COMPARISON: All models for each train->test pair
# ============================================================
for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        pair_key = (train_stock, test_stock)
        if pair_key not in all_predictions:
            continue
        
        y_true = all_predictions[pair_key][MODEL_TYPES[0]][0]
        dates = all_predictions[pair_key][MODEL_TYPES[0]][2]
        preds = {mt: all_predictions[pair_key][mt][1] for mt in MODEL_TYPES if mt in all_predictions[pair_key]}
        
        plot_all_models_comparison(
            dates, y_true, preds,
            f'Train_{train_stock}_Test_{test_stock}',
            EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )

print("All comparison plots saved!")


  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_TLKM_Test_BBCA_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_TLKM_Test_ASII_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_TLKM_Test_UNVR_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_BBCA_Test_TLKM_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_BBCA_Test_ASII_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_BBCA_Test_UNVR_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_ASII_Test_TLKM_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_ASII_Test_BBCA_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_ASII_Test_UNVR_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_UNVR_Test_TLKM_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_UNVR_Test_BBCA_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_UNVR_Test_ASII_all_models.png
All comparison p

In [ ]:
# ============================================================
# SUMMARY: BEST MODEL PER CROSS-STOCK PAIR
# ============================================================
print("\n" + "="*70)
print("  BEST MODEL PER PAIR (by RMSE)")
print("="*70)
for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        pair_data = results_df[
            (results_df['Train_Stock'] == train_stock) &
            (results_df['Test_Stock'] == test_stock)
        ]
        if pair_data.empty:
            continue
        best_idx = pair_data['RMSE'].idxmin()
        best = pair_data.loc[best_idx]
        print(f"  {train_stock} -> {test_stock}: {best['Model']} "
              f"(RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")



  BEST MODEL PER PAIR (by RMSE)
  TLKM -> BBCA: BiLSTM (RMSE=140.6905, R²=0.981687)
  TLKM -> ASII: GRU (RMSE=92.5174, R²=0.976832)
  TLKM -> UNVR: GRU (RMSE=89.1230, R²=0.990869)
  BBCA -> TLKM: GRU (RMSE=81.0537, R²=0.961544)
  BBCA -> ASII: GRU (RMSE=113.5532, R²=0.965099)
  BBCA -> UNVR: GRU (RMSE=111.9851, R²=0.985584)
  ASII -> TLKM: GRU (RMSE=59.4370, R²=0.979321)
  ASII -> BBCA: BiGRU (RMSE=132.8840, R²=0.983663)
  ASII -> UNVR: GRU (RMSE=83.2404, R²=0.992035)
  UNVR -> TLKM: BiGRU (RMSE=86.0327, R²=0.956674)
  UNVR -> BBCA: GRU (RMSE=142.7573, R²=0.981145)
  UNVR -> ASII: GRU (RMSE=102.8110, R²=0.971390)


## Interactive Visualization (Plotly)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# INTERACTIVE VISUALIZATION - ALL MODELS COMPARISON (Plotly)
# ============================================================
print("Generating interactive plots...")

for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        
        pair_key = (train_stock, test_stock)
        if pair_key not in all_predictions:
            continue
        
        print(f"  Generating interactive plot for {train_stock} → {test_stock}...")
        
        y_true = all_predictions[pair_key][MODEL_TYPES[0]][0]
        dates = all_predictions[pair_key][MODEL_TYPES[0]][2]
        
        # Create figure with all models visible
        fig = go.Figure()
        
        # Linestyles for variety
        linestyles_map = {
            'BiLSTM': 'dash',
            'BiGRU': 'dashdot',
            'LSTM': 'dot',
            'GRU': 'solid'
        }
        
        # Add actual values (always visible)
        fig.add_trace(go.Scatter(
            x=dates, y=y_true,
            name='Actual',
            mode='lines',
            line=dict(color='#808080', width=2.5, dash='solid'),
            hovertemplate='Date: %{x}<br>Price: %{y:.2f} IDR<extra></extra>',
            visible=True
        ))
        
        # Add all model predictions
        for model_type in MODEL_TYPES:
            if model_type in all_predictions[pair_key]:
                y_pred = all_predictions[pair_key][model_type][1]
                
                fig.add_trace(go.Scatter(
                    x=dates, y=y_pred,
                    name=model_type,
                    mode='lines',
                    line=dict(
                        color=MODEL_COLORS[model_type],
                        width=2,
                        dash=linestyles_map.get(model_type, 'solid')
                    ),
                    hovertemplate='Date: %{x}<br>Price: %{y:.2f} IDR<extra></extra>',
                    visible=True
                ))
        
        # Update layout
        fig.update_layout(
            title=f'<b>Cross-Stock Prediction: {train_stock} → {test_stock}</b><br><sub>Zero-Shot Transfer Learning</sub>',
            xaxis_title='Date',
            yaxis_title='Close Price (IDR)',
            hovermode='x unified',
            template='plotly_white',
            font=dict(size=12),
            height=600,
            width=1200,
            margin=dict(l=50, r=50, t=100, b=50)
        )
        
        # Add range slider
        fig.update_xaxes(rangeslider_visible=False)
        
        # Save ALL MODELS plot
        html_filename = f'figures/{EXP_LABEL}/interactive_{train_stock}_{test_stock}_all_models.html'
        fig.write_html(html_filename)
        print(f"    ✓ Saved: {html_filename}")

print("\n✓ All interactive plots generated and saved!")
print(f"  Location: figures/{EXP_LABEL}/interactive_*.html")


Generating interactive plots...
  Generating interactive plot for TLKM → BBCA...
    ✓ Saved: figures/Exp2_80_20/interactive_TLKM_BBCA_all_models.html
  Generating interactive plot for TLKM → ASII...
    ✓ Saved: figures/Exp2_80_20/interactive_TLKM_ASII_all_models.html
  Generating interactive plot for TLKM → UNVR...
    ✓ Saved: figures/Exp2_80_20/interactive_TLKM_UNVR_all_models.html
  Generating interactive plot for BBCA → TLKM...
    ✓ Saved: figures/Exp2_80_20/interactive_BBCA_TLKM_all_models.html
  Generating interactive plot for BBCA → ASII...
    ✓ Saved: figures/Exp2_80_20/interactive_BBCA_ASII_all_models.html
  Generating interactive plot for BBCA → UNVR...
    ✓ Saved: figures/Exp2_80_20/interactive_BBCA_UNVR_all_models.html
  Generating interactive plot for ASII → TLKM...
    ✓ Saved: figures/Exp2_80_20/interactive_ASII_TLKM_all_models.html
  Generating interactive plot for ASII → BBCA...
    ✓ Saved: figures/Exp2_80_20/interactive_ASII_BBCA_all_models.html
  Generating int

In [ ]:
# ============================================================
# INTERACTIVE VISUALIZATION - TOGGLE BUTTONS (Plotly)
# ============================================================
print("Generating interactive toggle plots...")

for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        
        pair_key = (train_stock, test_stock)
        if pair_key not in all_predictions:
            continue
        
        print(f"  Generating toggle plot for {train_stock} → {test_stock}...")
        
        y_true = all_predictions[pair_key][MODEL_TYPES[0]][0]
        dates = all_predictions[pair_key][MODEL_TYPES[0]][2]
        
        # Create figure with toggle buttons
        fig = go.Figure()
        
        linestyles_map = {
            'BiLSTM': 'dash',
            'BiGRU': 'dashdot',
            'LSTM': 'dot',
            'GRU': 'solid'
        }
        
        # Add actual values (always visible)
        fig.add_trace(go.Scatter(
            x=dates, y=y_true,
            name='Actual',
            mode='lines',
            line=dict(color='#808080', width=2.5, dash='solid'),
            hovertemplate='Date: %{x}<br>Price: %{y:.2f} IDR<extra></extra>',
            visible=True
        ))
        
        # Add each model with visibility control
        for idx, model_type in enumerate(MODEL_TYPES):
            if model_type in all_predictions[pair_key]:
                y_pred = all_predictions[pair_key][model_type][1]
                
                # First model visible by default, others hidden
                is_visible = True if idx == 0 else False
                
                fig.add_trace(go.Scatter(
                    x=dates, y=y_pred,
                    name=model_type,
                    mode='lines',
                    line=dict(
                        color=MODEL_COLORS[model_type],
                        width=2.5,
                        dash=linestyles_map.get(model_type, 'solid')
                    ),
                    hovertemplate='Date: %{x}<br>Price: %{y:.2f} IDR<extra></extra>',
                    visible=is_visible
                ))
        
        # Create buttons for toggling models
        buttons = []
        for i, model_type in enumerate(MODEL_TYPES):
            # Create visibility list: [True for Actual, False for all models except this one, True for this model]
            visibility = [True] + [j == i for j in range(len(MODEL_TYPES))]
            
            buttons.append(
                dict(
                    label=model_type,
                    method='update',
                    args=[
                        {'visible': visibility},
                        {'title': f'<b>Cross-Stock Prediction: {train_stock} → {test_stock}</b><br><sub>{model_type} Model</sub>'}
                    ]
                )
            )
        
        # Add "Show All" button
        buttons.insert(0, dict(
            label='All Models',
            method='update',
            args=[
                {'visible': [True] * (len(MODEL_TYPES) + 1)},
                {'title': f'<b>Cross-Stock Prediction: {train_stock} → {test_stock}</b><br><sub>All Models Comparison</sub>'}
            ]
        ))
        
        # Update layout with buttons
        fig.update_layout(
            updatemenus=[
                dict(
                    active=0,
                    buttons=buttons,
                    direction="down",
                    pad={"r": 10, "t": 10},
                    showactive=True,
                    x=0.0,
                    xanchor="left",
                    y=1.15,
                    yanchor="top",
                    bgcolor="#E0E0E0",
                    bordercolor="#555",
                    borderwidth=1
                )
            ],
            title=f'<b>Cross-Stock Prediction: {train_stock} → {test_stock}</b><br><sub>All Models Comparison</sub>',
            xaxis_title='Date',
            yaxis_title='Close Price (IDR)',
            hovermode='x unified',
            template='plotly_white',
            font=dict(size=12),
            height=600,
            width=1200,
            margin=dict(l=50, r=50, t=150, b=50)
        )
        
        # Save TOGGLE plot
        html_filename = f'figures/{EXP_LABEL}/interactive_{train_stock}_{test_stock}_toggle.html'
        fig.write_html(html_filename)
        print(f"    ✓ Saved: {html_filename}")

print("\n✓ All interactive toggle plots generated and saved!")
print(f"  Location: figures/{EXP_LABEL}/interactive_*_toggle.html")
